# Benchmark Multimodal — SecureBERT vs DomURLs-BERT

**Dataset:** Multimodal (URL + WHOIS + DOM)  
**Fonte:** dataset_phishing_brasileiro.csv + whois_cache.json + GregaVrbancic/Phishing-Dataset  
**Modelos:** SecureBERT (ehsanaghaei/SecureBERT) e DomURLs-BERT (amahdaouy/DomURLs_BERT)  
**GPU:** Colab T4 (fp16)  

Cenário: testar os dois modelos com input enriquecido (URL + metadados WHOIS + features DOM),  
não apenas URLs puras, para avaliar a capacidade de cada modelo em contextos mais completos.

In [ ]:
# ============================================================
# 1. Verificação de GPU e Instalação
# ============================================================
import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.mem_get_info()
    print(f"GPU: {gpu} | VRAM livre: {vram[0]/1e9:.1f} GB / {vram[1]/1e9:.1f} GB")
else:
    print("SEM GPU — o benchmark será extremamente lento!")

!pip install -q transformers datasets accelerate tldextract scikit-learn matplotlib seaborn

In [ ]:
# ============================================================
# 2. Imports
# ============================================================
import os, gc, json, time, math, warnings
from urllib.parse import urlparse

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tldextract

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, matthews_corrcoef,
    log_loss, confusion_matrix, roc_curve, precision_recall_curve,
    ConfusionMatrixDisplay,
)

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer,
)
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings('ignore', category=FutureWarning)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

In [ ]:
# ============================================================
# 3. Configuração Global
# ============================================================
RANDOM_STATE      = 42
MAX_TRAIN_SAMPLES = 200_000   # None = tudo
MAX_EVAL_SAMPLES  = 50_000    # None = tudo
MAX_LENGTH        = 192       # maior que 128 para acomodar tokens DOM/WHOIS
NUM_EPOCHS        = 3
LR                = 2e-5
WARMUP_RATIO      = 0.06
WEIGHT_DECAY      = 0.01

# Google Drive
DRIVE_BASE = "/content/drive/MyDrive/TCC-Phishing-Benchmark-Multimodal"

print(f"MAX_TRAIN   : {MAX_TRAIN_SAMPLES:,}")
print(f"MAX_EVAL    : {MAX_EVAL_SAMPLES:,}")
print(f"MAX_LENGTH  : {MAX_LENGTH} tokens")
print(f"NUM_EPOCHS  : {NUM_EPOCHS}")
print(f"DRIVE_BASE  : {DRIVE_BASE}")

In [ ]:
# ============================================================
# 4. Montar Google Drive e Criar Diretórios
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

for subdir in ['models', 'resultados', 'predicoes', 'graficos']:
    os.makedirs(f"{DRIVE_BASE}/{subdir}", exist_ok=True)
    print(f"  ✓ {DRIVE_BASE}/{subdir}")

In [ ]:
# ============================================================
# 5. Carregar / Criar Dataset Multimodal
# ============================================================
# Opção A: fazer upload do dataset_multimodal.csv já pronto
# Opção B: construir direto aqui a partir das fontes

MULTIMODAL_CSV = "/content/dataset_multimodal.csv"
if not os.path.exists(MULTIMODAL_CSV):
    MULTIMODAL_CSV = f"{DRIVE_BASE}/dataset_multimodal.csv"

if os.path.exists(MULTIMODAL_CSV):
    print(f"Carregando dataset pronto: {MULTIMODAL_CSV}")
    df = pd.read_csv(MULTIMODAL_CSV, encoding='utf-8')
else:
    print("Dataset não encontrado. Construindo a partir das fontes...")
    print("Isso pode levar alguns minutos na primeira vez.\n")

    # --- Fonte 1: Dataset brasileiro ---
    BR_CSV = "/content/dataset_phishing_brasileiro.csv"
    if not os.path.exists(BR_CSV):
        BR_CSV = f"{DRIVE_BASE}/dataset_phishing_brasileiro.csv"
    print(f"Carregando dataset brasileiro: {BR_CSV}")
    br_df = pd.read_csv(BR_CSV, encoding='utf-8', usecols=['url', 'label'])
    print(f"  {len(br_df):,} amostras")

    # --- Fonte 2: WHOIS cache ---
    WHOIS_PATH = "/content/whois_cache.json"
    if not os.path.exists(WHOIS_PATH):
        WHOIS_PATH = f"{DRIVE_BASE}/whois_cache.json"
    whois = {}
    if os.path.exists(WHOIS_PATH):
        with open(WHOIS_PATH, encoding='utf-8') as f:
            whois = json.load(f)
        print(f"  WHOIS: {len(whois):,} domínios")
    else:
        print("  WHOIS: não encontrado, rodando sem WHOIS")

    # --- Fonte 3: GregaVrbancic DOM features ---
    DOM_URL = "https://raw.githubusercontent.com/GregaVrbancic/Phishing-Dataset/master/dataset_full.csv"
    print(f"Baixando dataset DOM (GregaVrbancic)...")
    dom_df = pd.read_csv(DOM_URL)
    print(f"  {len(dom_df):,} amostras, {len(dom_df.columns)} colunas")

    # Features DOM selecionadas
    GREGA_FEATURES = [
        'qty_redirects', 'length_url', 'domain_length', 'qty_dot_url',
        'qty_hyphen_url', 'qty_slash_url', 'qty_at_url', 'qty_params',
        'url_shortened', 'tls_ssl_certificate', 'time_domain_activation',
        'time_domain_expiration', 'qty_mx_servers', 'qty_nameservers',
        'domain_google_index', 'domain_spf', 'domain_in_ip',
        'qty_vowels_domain', 'server_client_domain', 'email_in_url',
    ]
    grega_feats = [f for f in GREGA_FEATURES if f in dom_df.columns]
    print(f"  Features disponíveis: {len(grega_feats)}/{len(GREGA_FEATURES)}")

    # --- Funções auxiliares ---
    def extract_domain(url):
        try:
            ext = tldextract.extract(url)
            return f"{ext.domain}.{ext.suffix}" if ext.suffix else ext.domain
        except Exception:
            return ""

    def format_whois(url, cache):
        domain = extract_domain(url)
        info = cache.get(domain, {})
        if info.get('status') == 'found':
            age = info.get('domain_age_days', '?')
            reg = str(info.get('registrar', 'unk'))[:25].strip()
            expire = info.get('days_to_expire', '?')
            return f"[AGE] {age}d [REG] {reg} [EXPIRE] {expire}d [WHOIS] found"
        return "[WHOIS] unknown"

    def format_extra(row, feats):
        parts = []
        for f in feats:
            val = row.get(f, None)
            if val is not None and not (isinstance(val, float) and math.isnan(val)):
                short = (f.replace('qty_', '').replace('_url', '')
                          .replace('domain_', 'dom_').replace('server_client_domain', 'srv_client')
                          .replace('tls_ssl_certificate', 'tls')
                          .replace('time_domain_activation', 'dom_age')
                          .replace('time_domain_expiration', 'dom_expire')
                          .replace('url_shortened', 'shortened').replace('email_in_url', 'email'))
                parts.append(f"{short}={int(val)}")
        return '[EXTRA] ' + ' '.join(parts) if parts else '[EXTRA] none'

    # --- Processar GregaVrbancic (sem coluna url — ID sintético) ---
    print("
Processando dataset GregaVrbancic...")
    grega_records = []
    for idx, row in dom_df.iterrows():
        synth_url = f"grega_sample_{idx}"
        label = int(row.get('phishing', 0))
        e = format_extra(row, grega_feats)
        grega_records.append({'url': synth_url, 'label': label, 'text_input': f'[URL] {synth_url} [WHOIS] unknown {e}', 'source': 'grega'})
    grega_proc = pd.DataFrame(grega_records)
    print(f"  GregaVrbancic: {len(grega_proc):,}")

    # --- Processar brasileiro ---
    print("Processando dataset brasileiro...")
    br_records = []
    for _, row in br_df.iterrows():
        url = str(row['url'])
        label = int(row['label'])
        w = format_whois(url, whois)
        br_records.append({'url': url, 'label': label, 'text_input': f'[URL] {url} {w} [EXTRA] none', 'source': 'brasileiro'})
    br_proc = pd.DataFrame(br_records)
    print(f"  Brasileiro: {len(br_proc):,}")

    # --- Combinar ---
    df = pd.concat([grega_proc, br_proc], ignore_index=True)
    df = df.drop_duplicates(subset=['url'], keep='first')
    df = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
    df.to_csv(f"{DRIVE_BASE}/dataset_multimodal.csv", index=False, encoding='utf-8')
    print(f"\nDataset salvo no Drive: {len(df):,} amostras")

print(f"\nDataset multimodal: {len(df):,} amostras")
print(f"Phishing: {df['label'].mean():.1%}")
print(f"GregaVrbancic: {(df['source'] == 'grega').sum():,}")
print(f"Brasileiro: {(df['source'] == 'brasileiro').sum():,}")
print(f"\nExemplo GregaVrbancic:")
print(f"  {df[df['source']=='grega'].iloc[0]['text_input'][:200]}")
print(f"\nExemplo Brasileiro:")
print(f"  {df[df['source']=='brasileiro'].iloc[0]['text_input'][:200]}")

In [ ]:
# ============================================================
# 6. Split Treino / Avaliação
# ============================================================
train_df, eval_df = train_test_split(
    df, test_size=0.2, stratify=df['label'], random_state=RANDOM_STATE
)
print(f"Split: treino={len(train_df):,} | avaliação={len(eval_df):,}")

# Amostragem se necessário
if MAX_TRAIN_SAMPLES and len(train_df) > MAX_TRAIN_SAMPLES:
    frac = MAX_TRAIN_SAMPLES / len(train_df)
    train_df = (
        train_df.groupby('label', group_keys=False)
        .apply(lambda x: x.sample(frac=frac, random_state=RANDOM_STATE))
        .reset_index(drop=True)
    )
    print(f"Amostra treino (estratificada): {len(train_df):,}")

if MAX_EVAL_SAMPLES and len(eval_df) > MAX_EVAL_SAMPLES:
    frac = MAX_EVAL_SAMPLES / len(eval_df)
    eval_df = (
        eval_df.groupby('label', group_keys=False)
        .apply(lambda x: x.sample(frac=frac, random_state=RANDOM_STATE))
        .reset_index(drop=True)
    )
    print(f"Amostra avaliação (estratificada): {len(eval_df):,}")

train_texts  = train_df['text_input'].tolist()
eval_texts   = eval_df['text_input'].tolist()
train_labels = train_df['label'].astype(int).tolist()
eval_labels  = eval_df['label'].astype(int).tolist()

print(f"\nTreino: {len(train_texts):,} | Avaliação: {len(eval_texts):,}")
print(f"Phishing treino: {sum(train_labels)/len(train_labels):.1%}")
print(f"Phishing eval:   {sum(eval_labels)/len(eval_labels):.1%}")

In [ ]:
# ============================================================
# 7. Classes e Funções Compartilhadas
# ============================================================

class URLDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.encodings = tokenizer(
            texts, truncation=True, padding='max_length',
            max_length=max_length, return_tensors='pt',
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item


def compute_hf_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': float(accuracy_score(labels, preds)),
        'f1':       float(f1_score(labels, preds, zero_division=0)),
    }


def compute_full_metrics(y_true, y_pred, y_proba, model_name, train_time, latencies):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        'Modelo':                  model_name,
        'Accuracy':                float(accuracy_score(y_true, y_pred)),
        'Precision':               float(precision_score(y_true, y_pred, zero_division=0)),
        'Recall':                  float(recall_score(y_true, y_pred, zero_division=0)),
        'F1-Score':                float(f1_score(y_true, y_pred, zero_division=0)),
        'AUC-ROC':                 float(roc_auc_score(y_true, y_proba)),
        'Avg Precision (PR-AUC)':  float(average_precision_score(y_true, y_proba)),
        'MCC':                     float(matthews_corrcoef(y_true, y_pred)),
        'Log Loss':                float(log_loss(y_true, y_proba)),
        'Specificity':             float(tn / (tn + fp)),
        'Latência P50 (ms)':       float(np.percentile(latencies, 50)),
        'Latência P95 (ms)':       float(np.percentile(latencies, 95)),
        'Latência P99 (ms)':       float(np.percentile(latencies, 99)),
        'Tempo Treino (s)':        float(train_time),
    }


def run_model(cfg):
    """
    Treina (ou carrega do cache) e avalia um modelo.
    cfg: name, model_id, batch_size, grad_accum
    """
    name       = cfg['name']
    model_id   = cfg['model_id']
    bs         = cfg['batch_size']
    ga         = cfg['grad_accum']
    seq_len    = cfg.get('max_length', MAX_LENGTH)

    model_path = f"{DRIVE_BASE}/models/{name}"
    preds_file = f"{DRIVE_BASE}/predicoes/{name}_predictions.npz"

    model_exists = os.path.exists(os.path.join(model_path, 'config.json'))
    preds_exist  = os.path.exists(preds_file)

    print(f"\n{'='*65}")
    print(f"Modelo : {name}")
    print(f"ID     : {model_id}")
    print(f"Config : batch={bs} | grad_accum={ga} | efetivo={bs*ga} | max_length={seq_len}")
    print(f"Cache  : modelo={'OK' if model_exists else 'X'} | predicoes={'OK' if preds_exist else 'X'}")
    print(f"{'='*65}")

    train_time = 0.0

    # ── Treinar ou carregar ─────────────────────────────────
    if model_exists:
        print("  Carregando modelo do Drive (treino pulado)...")
        tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=True)
        model     = AutoModelForSequenceClassification.from_pretrained(model_path, num_labels=2)
    else:
        print(f"  Carregando tokenizer e modelo base: {model_id}")
        tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
        train_ds  = URLDataset(train_texts, train_labels, tokenizer, seq_len)
        eval_ds   = URLDataset(eval_texts,  eval_labels,  tokenizer, seq_len)

        model = AutoModelForSequenceClassification.from_pretrained(
            model_id, num_labels=2, ignore_mismatched_sizes=True,
        )
        n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"  Parâmetros treináveis: {n_params/1e6:.1f}M")

        training_args = TrainingArguments(
            output_dir=f'/content/tmp_{name}',
            num_train_epochs=NUM_EPOCHS,
            per_device_train_batch_size=bs,
            per_device_eval_batch_size=bs * 2,
            gradient_accumulation_steps=ga,
            learning_rate=LR,
            warmup_ratio=WARMUP_RATIO,
            weight_decay=WEIGHT_DECAY,
            logging_steps=500,
            eval_strategy='epoch',
            save_strategy='no',
            load_best_model_at_end=False,
            fp16=torch.cuda.is_available(),
            bf16=False,
            report_to='none',
            seed=RANDOM_STATE,
            dataloader_num_workers=2,
            dataloader_pin_memory=True,
        )
        trainer = Trainer(
            model=model, args=training_args,
            train_dataset=train_ds, eval_dataset=eval_ds,
            compute_metrics=compute_hf_metrics,
        )

        print(f"  Iniciando treino ({NUM_EPOCHS} épocas)...")
        t_start = time.time()
        trainer.train()
        train_time = time.time() - t_start
        print(f"  Treino concluído: {train_time/60:.1f} min")

        os.makedirs(model_path, exist_ok=True)
        model.save_pretrained(model_path)
        tokenizer.save_pretrained(model_path)
        print(f"  Modelo salvo: {model_path}")

        del trainer, train_ds, eval_ds
        gc.collect()
        torch.cuda.empty_cache()

    # ── Predições em batch ──────────────────────────────────
    if preds_exist:
        print("  Carregando predições do Drive...")
        data       = np.load(preds_file)
        y_true     = data['y_true']
        y_pred     = data['y_pred']
        y_proba    = data['y_proba']
        train_time = float(data['train_time'][0]) if 'train_time' in data else train_time
    else:
        print("  Gerando predições em batch...")
        model_gpu = model.to(device)
        model_gpu.eval()
        eval_ds_pred = URLDataset(eval_texts, eval_labels, tokenizer, seq_len)
        loader = DataLoader(eval_ds_pred, batch_size=bs * 2, shuffle=False, pin_memory=True)

        all_logits = []
        with torch.no_grad():
            for batch in loader:
                inputs  = {k: v.to(device) for k, v in batch.items() if k != 'labels'}
                outputs = model_gpu(**inputs)
                all_logits.append(outputs.logits.cpu().numpy())

        logits  = np.vstack(all_logits)
        y_proba = torch.softmax(torch.tensor(logits, dtype=torch.float32), dim=-1)[:, 1].numpy()
        y_pred  = np.argmax(logits, axis=-1)
        y_true  = np.array(eval_labels)

        np.savez(
            preds_file,
            y_true=y_true, y_pred=y_pred, y_proba=y_proba,
            train_time=np.array([train_time]),
        )
        print(f"  Predições salvas: {preds_file}")

        del eval_ds_pred, loader, all_logits, model_gpu
        gc.collect()
        torch.cuda.empty_cache()

    # ── Latência GPU single-sample ──────────────────────────
    print("  Medindo latência GPU (500 amostras individuais)...")
    model_gpu = model.to(device)
    model_gpu.eval()
    latencies = []

    # Warmup
    for text in eval_texts[:5]:
        enc = tokenizer([text], truncation=True, padding='max_length', max_length=seq_len, return_tensors='pt')
        with torch.no_grad():
            _ = model_gpu(**{k: v.to(device) for k, v in enc.items()})

    for text in eval_texts[:500]:
        enc    = tokenizer([text], truncation=True, padding='max_length', max_length=seq_len, return_tensors='pt')
        inputs = {k: v.to(device) for k, v in enc.items()}
        if device.type == 'cuda':
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.no_grad():
            _ = model_gpu(**inputs)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        latencies.append((time.perf_counter() - t0) * 1000)

    latencies = np.array(latencies)

    # ── Métricas ────────────────────────────────────────────
    row = compute_full_metrics(y_true, y_pred, y_proba, name, train_time, latencies)

    print(f"\n  {'Accuracy':<22}: {row['Accuracy']:.4f}")
    print(f"  {'Precision':<22}: {row['Precision']:.4f}")
    print(f"  {'Recall':<22}: {row['Recall']:.4f}")
    print(f"  {'F1-Score':<22}: {row['F1-Score']:.4f}")
    print(f"  {'AUC-ROC':<22}: {row['AUC-ROC']:.4f}")
    print(f"  {'PR-AUC':<22}: {row['Avg Precision (PR-AUC)']:.4f}")
    print(f"  {'MCC':<22}: {row['MCC']:.4f}")
    print(f"  {'Latência P50':<22}: {row['Latência P50 (ms)']:.2f} ms")
    print(f"  {'Latência P95':<22}: {row['Latência P95 (ms)']:.2f} ms")
    treino_str = f"{train_time/60:.1f} min" if train_time > 0 else 'carregado do cache'
    print(f"  {'Tempo treino':<22}: {treino_str}")

    csv_path = f"{DRIVE_BASE}/resultados/resultado_{name}_multimodal.csv"
    pd.DataFrame([row]).set_index('Modelo').to_csv(csv_path)
    print(f"  Resultado CSV salvo: {csv_path}")

    del model, model_gpu
    gc.collect()
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        print(f"  VRAM livre: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

    return row, {'y_pred': y_pred, 'y_proba': y_proba, 'y_true': y_true}


results         = []
all_predictions = {}
print("Funções definidas. Pronto para rodar os modelos.")

---
## Modelos — execute cada célula individualmente

Se a sessão cair, remonte o Drive (célula 4) e recarregue o dataset (célula 5).  
Os modelos e predições são salvos no Drive para cache automático.

In [ ]:
# ============================================================
# MODELO 1 — DomURLs-BERT (amahdaouy/DomURLs_BERT)
# ~110M parâmetros | Pré-treinado em URLs e domínios maliciosos
# Vocabulário customizado de 32k tokens focado em URLs
# Hipótese: deve se beneficiar dos tokens [URL] e [EXTRA]
# ============================================================
_row, _preds = run_model({
    'name':       'DomURLs-BERT-multimodal',
    'model_id':   'amahdaouy/DomURLs_BERT',
    'batch_size': 64,
    'grad_accum': 1,
})
results.append(_row)
all_predictions['DomURLs-BERT-multimodal'] = _preds

In [ ]:
# ============================================================
# MODELO 2 — SecureBERT (ehsanaghaei/SecureBERT)
# ~125M parâmetros | RoBERTa-base pré-treinado em corpus cyber
# Hipótese: vocabulário cyber amplo pode capturar padrões WHOIS
# ============================================================
_row, _preds = run_model({
    'name':       'SecureBERT-multimodal',
    'model_id':   'ehsanaghaei/SecureBERT',
    'batch_size': 64,
    'grad_accum': 1,
})
results.append(_row)
all_predictions['SecureBERT-multimodal'] = _preds

---
## Análise e Visualizações

In [ ]:
# ============================================================
# 8. Tabela Comparativa
# ============================================================
if not results:
    # Tentar carregar do Drive
    import glob
    csv_files = sorted(glob.glob(f"{DRIVE_BASE}/resultados/resultado_*_multimodal.csv"))
    for f in csv_files:
        row = pd.read_csv(f).iloc[0].to_dict()
        results.append(row)
    print(f"Carregados {len(results)} resultados do Drive")

results_df = pd.DataFrame(results).set_index('Modelo')

display_cols = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC',
                'MCC', 'Latência P50 (ms)', 'Tempo Treino (s)']
print("\n" + "=" * 80)
print("BENCHMARK MULTIMODAL — SecureBERT vs DomURLs-BERT")
print("=" * 80)
display(results_df[display_cols].round(4))

results_df.to_csv(f"{DRIVE_BASE}/resultados/tabela_comparativa_multimodal.csv")
print(f"\nTabela salva: {DRIVE_BASE}/resultados/tabela_comparativa_multimodal.csv")

In [ ]:
# ============================================================
# 9. Matrizes de Confusão
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, (name, preds) in enumerate(all_predictions.items()):
    cm = confusion_matrix(preds['y_true'], preds['y_pred'])
    disp = ConfusionMatrixDisplay(cm, display_labels=['Legítimo', 'Phishing'])
    disp.plot(ax=axes[i], cmap='Blues', values_format='d')
    axes[i].set_title(name, fontsize=13)

plt.suptitle('Matrizes de Confusão — Benchmark Multimodal', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig(f"{DRIVE_BASE}/graficos/confusion_matrices_multimodal.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 10. Curvas ROC e Precision-Recall
# ============================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
colors = ['#2196F3', '#FF5722']

for i, (name, preds) in enumerate(all_predictions.items()):
    # ROC
    fpr, tpr, _ = roc_curve(preds['y_true'], preds['y_proba'])
    auc_val = roc_auc_score(preds['y_true'], preds['y_proba'])
    ax1.plot(fpr, tpr, color=colors[i], lw=2, label=f"{name} (AUC={auc_val:.4f})")

    # PR
    prec, rec, _ = precision_recall_curve(preds['y_true'], preds['y_proba'])
    ap = average_precision_score(preds['y_true'], preds['y_proba'])
    ax2.plot(rec, prec, color=colors[i], lw=2, label=f"{name} (AP={ap:.4f})")

ax1.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
ax1.set_xlabel('FPR')
ax1.set_ylabel('TPR')
ax1.set_title('Curva ROC')
ax1.legend(loc='lower right')
ax1.grid(True, alpha=0.3)

ax2.set_xlabel('Recall')
ax2.set_ylabel('Precision')
ax2.set_title('Curva Precision-Recall')
ax2.legend(loc='lower left')
ax2.grid(True, alpha=0.3)

plt.suptitle('Curvas ROC e PR — Benchmark Multimodal', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig(f"{DRIVE_BASE}/graficos/roc_pr_curves_multimodal.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 11. Comparação de Métricas (Barras)
# ============================================================
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC', 'MCC']
plot_df = results_df[metrics_to_plot].T

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(metrics_to_plot))
width = 0.35
models = plot_df.columns.tolist()

bars1 = ax.bar(x - width/2, plot_df[models[0]], width, label=models[0], color='#2196F3', alpha=0.85)
bars2 = ax.bar(x + width/2, plot_df[models[1]], width, label=models[1], color='#FF5722', alpha=0.85)

ax.set_ylabel('Score')
ax.set_title('Comparação de Métricas — Benchmark Multimodal')
ax.set_xticks(x)
ax.set_xticklabels(metrics_to_plot)
ax.legend()
ax.set_ylim(0, 1.05)
ax.grid(axis='y', alpha=0.3)

for bar in bars1:
    ax.annotate(f'{bar.get_height():.3f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    ax.annotate(f'{bar.get_height():.3f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig(f"{DRIVE_BASE}/graficos/metrics_comparison_multimodal.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 12. Comparação com Benchmark URL-only (se disponível)
# ============================================================
URL_ONLY_BASE = "/content/drive/MyDrive/TCC-Phishing-Benchmark"

url_only_results = []
for model_name in ['DomURLs-BERT', 'SecureBERT']:
    csv_path = f"{URL_ONLY_BASE}/resultados/resultado_{model_name}.csv"
    if os.path.exists(csv_path):
        row = pd.read_csv(csv_path).iloc[0].to_dict()
        row['Modelo'] = f"{model_name} (URL-only)"
        url_only_results.append(row)

if url_only_results:
    all_results = results + url_only_results
    compare_df = pd.DataFrame(all_results).set_index('Modelo')

    compare_cols = ['F1-Score', 'AUC-ROC', 'MCC', 'Latência P50 (ms)']
    print("\n" + "=" * 80)
    print("COMPARAÇÃO: Multimodal vs URL-only")
    print("=" * 80)
    display(compare_df[compare_cols].round(4))

    # Gráfico de comparação
    fig, ax = plt.subplots(figsize=(12, 6))
    plot_cols = ['F1-Score', 'AUC-ROC', 'MCC']
    comp_plot = compare_df[plot_cols].T
    comp_plot.plot(kind='bar', ax=ax, alpha=0.85, width=0.7)
    ax.set_ylabel('Score')
    ax.set_title('Multimodal vs URL-only — SecureBERT e DomURLs-BERT')
    ax.set_ylim(0, 1.05)
    ax.grid(axis='y', alpha=0.3)
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig(f"{DRIVE_BASE}/graficos/multimodal_vs_urlonly.png", dpi=150, bbox_inches='tight')
    plt.show()

    compare_df.to_csv(f"{DRIVE_BASE}/resultados/comparacao_multimodal_vs_urlonly.csv")
    print(f"Tabela salva: {DRIVE_BASE}/resultados/comparacao_multimodal_vs_urlonly.csv")
else:
    print("Resultados URL-only não encontrados no Drive.")
    print(f"Esperado em: {URL_ONLY_BASE}/resultados/resultado_DomURLs-BERT.csv")
    print("Execute o benchmark_modelos_grandes.ipynb primeiro para ter a comparação.")

In [ ]:
# ============================================================
# 13. Análise por Subconjunto (GregaVrbancic vs Brasileiro)
# ============================================================
# Avaliar se os modelos performam melhor nas amostras que TÊM features extras (GregaVrbancic)
eval_sources = eval_df['source'].values

print("\n" + "=" * 80)
print("ANÁLISE POR SUBCONJUNTO: amostras GregaVrbancic vs Brasileiro")
print("=" * 80)

for name, preds in all_predictions.items():
    y_true = preds['y_true']
    y_pred = preds['y_pred']
    y_prob = preds['y_proba']

    for src_name, src_label in [('GregaVrbancic', 'grega'), ('Brasileiro', 'brasileiro')]:
        mask = eval_sources == src_label
        if mask.sum() == 0:
            continue
        yt = y_true[mask]
        yp = y_pred[mask]
        yb = y_prob[mask]
        f1  = f1_score(yt, yp, zero_division=0)
        auc = roc_auc_score(yt, yb) if len(np.unique(yt)) > 1 else 0
        mcc = matthews_corrcoef(yt, yp)
        print(f"  {name} | {src_name}: F1={f1:.4f} | AUC-ROC={auc:.4f} | MCC={mcc:.4f} | n={mask.sum():,}")
    print()

In [ ]:
# ============================================================
# 14. Exportar Tabela Final
# ============================================================
print("\nBenchmark multimodal concluído!")
print(f"Resultados em: {DRIVE_BASE}/resultados/")
print(f"Gráficos em:   {DRIVE_BASE}/graficos/")
print(f"Modelos em:    {DRIVE_BASE}/models/")